# Lesson 12 - 使用代理備忘錄減少聊天歷史記錄

本筆記本演示了如何使用 Microsoft 代理框架管理長對話中的上下文。隨著對話增長，令牌數量也會增加—最終超出模型的上下文視窗。我們透過**上下文摘要模式**和用於持久記憶的**代理備忘錄**來解決此問題。

## 你將學到的內容：
1. **為什麼上下文管理很重要**：理解令牌限制和上下文視窗
2. **上下文感知代理**：構建管理自身對話上下文的代理
3. **上下文摘要模式**：使用工具濃縮對話歷史
4. **代理備忘錄**：持久記憶，可存活於上下文減少過程

## 預備知識：
- 已設定好 Azure OpenAI 及環境變數
- 了解前面課程中的基本代理概念


## 設置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## 為什麼語境管理重要

每個大型語言模型 (LLM) 都有有限的**語境視窗**——它能夠在一次請求中處理的最大標記數。隨著多輪對話進行：

- **標記數隨每條用戶訊息和助理回覆線性增長**。
- **提示標記佔主導成本**，因為整個歷史每一輪都會重新傳送。
- 最終對話會**超出語境視窗**，模型要麼截斷內容，要麼報錯。

### 語境管理策略

| 策略 | 運作方式 | 權衡 |
|---|---|---|
| **截斷** | 捨棄最舊訊息 | 失去早期語境 |
| **摘要** | 將舊訊息濃縮為摘要 | 捨棄部分細節，但保留關鍵點 |
| **草稿簿 / 外部記憶** | 將關鍵資訊存於對話之外 | 需要工具調用，但可承受任意縮減 |

在本筆記本中，我們結合了**摘要**與**草稿簿工具**，讓代理即使在對話歷史被濃縮時也能保持連貫性。


## 建立具情境感知的代理程式


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## 模擬長時間對話

讓我們通過一個多輪對話來看看上下文如何累積。代理人應該在多輪中保留關鍵細節（偏好、預算、旅行日期），並展示連貫性。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

注意代理如何保留之前對話的上下文 — 它知道關於日本、壽司、寺廟、攝影、3000 美元預算、單人旅行以及四月的時間範圍。在短對話中這運作良好，但隨著對話增長，完整歷史會變得昂貴來重新傳送。

讓我們繼續對話，透過更多回合來觀察上下文的累積：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 上下文摘要模式

隨著對話的增加，我們可以使用 **摘要工具** 將累積的上下文壓縮成簡潔的格式。代理會調用此工具來記錄關鍵偏好，以便即使較舊的訊息被刪除，重要資訊仍能被保存。

此模式是更複雜歷史資料精簡的基礎：
1. 代理從對話中識別關鍵事實
2. 調用摘要工具來保存這些資訊
3. 可以安全地刪除較舊的訊息，因為摘要已涵蓋重要內容

以下定義了一個 `summarize_preferences` 工具，代理可以調用它來記錄所學資訊的簡潔摘要。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 總結

在此課程中，您學會了如何使用 Microsoft Agent Framework 來管理長期代理對話中的上下文：

### 主要概念
- **上下文視窗是有限的** — 對話歷史中的每個標記都會產生成本並計入限制。
- **摘要工具** 可讓代理將累積的上下文壓縮成簡潔的摘要，減少標記使用量，同時保留重要資訊。
- **代理便條本** 提供持久的外部記憶，能夠保存任何對話的縮減內容。

### 您所建立的內容
- 一個 **具備上下文感知能力的代理**，能在多輪對話中保持連續性
- 一個 **摘要工具** (`summarize_preferences`)，以簡潔格式記錄用戶的主要細節
- 一個 **多輪對話示範**，展示上下文保持與變更處理

### 實際應用
- **客戶服務機器人**：在長時間的支援會話中記住偏好設定
- **個人助理**：追蹤進行中的專案，無需重複解釋上下文
- **教育輔導員**：在多次互動中維持學生進度

### 下一步
- 實作具備檔案持久性的完整便條本工具
- 在摘要後新增自動歷史截斷機制
- 與向量資料庫結合以實現語義記憶搜尋
- 建立可於數日後恢復完整上下文的代理繼續對話


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件乃使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 所翻譯。雖然我們致力於保持準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件之母語版本應視為具權威性的資料來源。對於重要資訊，建議聘請專業人員進行人工翻譯。我們不對因使用本翻譯所引起之任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
